# Motor Temperature Prediction
## Data Preprocessing and Feature Engineering




In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib

## 1. Load the data


In [7]:
from google.colab import drive
import pandas as pd
import os

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/measures_v2.csv'

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
df.head()

Mounted at /content/drive
Dataset shape: (1330816, 13)


,u_q,coolant,stator_winding,u_d,stator_tooth,motor_speed,i_d,i_q,pm,stator_yoke,ambient,torque,profile_id
0,-0.450682,18.805172,19.086670,-0.350055,18.293219,0.002866,0.004419,0.000328,24.554214,18.316547,19.850691,0.187101,17
1,-0.325737,18.818571,19.092390,-0.305803,18.294807,0.000257,0.000606,-0.000785,24.538078,18.314955,19.850672,0.245417,17
2,-0.440864,18.828770,19.089380,-0.372503,18.294094,0.002355,0.001290,0.000386,24.544693,18.326307,19.850657,0.176615,17
3,-0.327026,18.835567,19.083031,-0.316199,18.292542,0.006105,0.000026,0.002046,24.554018,18.330833,19.850647,0.238303,17
4,-0.471150,18.857033,19.082525,-0.332272,18.291428,0.003133,-0.064317,0.037184,24.565397,18.326662,19.850639,0.208197,17


## 2. Feature engineering: current_magnitude
`i_d` and `i_q` are the two parts of the motor current. The heat in the winding depends on the
**total** current, so we combine them into one feature:

current_magnitude = sqrt(i_d² + i_q²)



In [8]:
df["current_magnitude"] = np.sqrt(df["i_d"]**2 + df["i_q"]**2)
df[["i_d", "i_q", "current_magnitude"]].head()

,i_d,i_q,current_magnitude
0,0.004419,0.000328,0.004431
1,0.000606,-0.000785,0.000992
2,0.001290,0.000386,0.001346
3,0.000026,0.002046,0.002046
4,-0.064317,0.037184,0.074292


## 3. Select features and target
- **Target:** `stator_winding` (the temperature we want to predict)


In [9]:
target = "stator_winding"
removed = ["stator_tooth", "stator_yoke", "pm"]

y = df[target]
X = df.drop(columns=removed + [target])

print(X.columns.tolist())

['u_q', 'coolant', 'u_d', 'motor_speed', 'i_d', 'i_q', 'ambient', 'torque', 'profile_id', 'current_magnitude']


## 4. Handling profile_id
`profile_id` is kept strictly for grouping splits and to prevent session overlap.

In [10]:
use_profile_id = False   # change to True to one-hot encode profile_id

groups = df["profile_id"]   # we need this for the split

if not use_profile_id:
    X = X.drop(columns=["profile_id"])

print(X.columns.tolist())

['u_q', 'coolant', 'u_d', 'motor_speed', 'i_d', 'i_q', 'ambient', 'torque', 'current_magnitude']


## 5. Train / test split


In [11]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

Train rows: 1026423
Test rows: 304393


## 6. Leakage check


In [12]:
train_profiles = set(groups.iloc[train_idx])
test_profiles = set(groups.iloc[test_idx])

print("Profiles in train:", len(train_profiles))
print("Profiles in test:", len(test_profiles))
print("Profiles in both:", len(train_profiles & test_profiles))

# this will stop the code with an error if there is any overlap
assert len(train_profiles & test_profiles) == 0

Profiles in train: 55
Profiles in test: 14
Profiles in both: 0


## 7. Scaling (and encoding)
The columns have very different ranges (for example `motor_speed` goes up to 6000 but `u_d` is much smaller).
Linear, Ridge and Lasso models are affected by this, so we use `StandardScaler`


In [13]:
if use_profile_id:
    # profile_id gets one-hot encoded, the other columns get scaled
    # (unseen profiles in test are ignored)
    num_cols = [c for c in X.columns if c != "profile_id"]
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["profile_id"])
    ])
else:
    # all remaining columns are numbers, so just scale them
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), X.columns.tolist())
    ])

In [14]:
# fit on train only, then transform both
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

col_names = preprocessor.get_feature_names_out()
X_train_ready = pd.DataFrame(X_train_ready, columns=col_names)
X_test_ready = pd.DataFrame(X_test_ready, columns=col_names)

X_train_ready.head()

,num__u_q,num__coolant,num__u_d,num__motor_speed,num__i_d,num__i_q,num__ambient,num__torque,num__current_magnitude
0,-1.206904,-0.815788,0.358828,-1.151504,1.053184,-0.391397,-2.384524,-0.385690,-1.251292
1,-1.204087,-0.815168,0.359533,-1.151506,1.053126,-0.391409,-2.384534,-0.384936,-1.251332
2,-1.206682,-0.814697,0.358470,-1.151505,1.053136,-0.391396,-2.384542,-0.385825,-1.251328
3,-1.204116,-0.814383,0.359368,-1.151503,1.053117,-0.391378,-2.384547,-0.385028,-1.251320
4,-1.207365,-0.813390,0.359111,-1.151504,1.052125,-0.390999,-2.384551,-0.385417,-1.250474


In [15]:
print("Train mean (first 5 columns):")
print(X_train_ready.mean().round(3).head())
print()
print("Train std (first 5 columns):")
print(X_train_ready.std().round(3).head())

Train mean (first 5 columns):
num__u_q            0.0
num__coolant       -0.0
num__u_d            0.0
num__motor_speed    0.0
num__i_d            0.0
dtype: float64

Train std (first 5 columns):
num__u_q            1.0
num__coolant        1.0
num__u_d            1.0
num__motor_speed    1.0
num__i_d            1.0
dtype: float64


## 8. Check for multicollinearity
In the EDA we saw that `i_q` and `torque` are almost the same (correlation 0.996).
`current_magnitude` is also made from `i_d` and `i_q`.

In [16]:
print(X_train[["i_d", "i_q", "torque", "current_magnitude"]].corr().round(3))

                     i_d    i_q  torque  current_magnitude
i_d                1.000 -0.221  -0.262             -0.845
i_q               -0.221  1.000   0.997              0.378
torque            -0.262  0.997   1.000              0.401
current_magnitude -0.845  0.378   0.401              1.000


## 9. Save files for the next persons

In [17]:
X_train_ready.to_csv("X_train.csv", index=False)
X_test_ready.to_csv("X_test.csv", index=False)
y_train.reset_index(drop=True).to_csv("y_train.csv", index=False)
y_test.reset_index(drop=True).to_csv("y_test.csv", index=False)

# Person 4 needs this for cross-validation (GroupKFold)
groups_train.reset_index(drop=True).to_csv("groups_train.csv", index=False)

# save the preprocessor so it can be reused on new data
joblib.dump(preprocessor, "preprocessor.pkl")

print("All files saved")

All files saved
